# ETL Basics: Extract → Transform → Aggregate → Load

## What you'll practice

Build a complete small ETL by hand: streaming extract with lineage columns, a strict transform that raises on bad data, rejected-record handling,
aggregation, and output files with run statistics. Code quality starts to matter here: small functions, type hints, docstrings, no hidden globals.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/etl/test_08_01_etl_basics.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/08_data_engineering_python/01_etl_basics.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Extract with lineage

**Implement:** `extract(path) -> Iterator[dict]`

Stream rows from a CSV as dicts (BOM-safe), adding `_source` (file name) and `_line` (physical line number from `reader.line_num`). Skip fully blank rows.

**Input:** `path`

**Expected output:** `Iterator[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_extract` in `tests/etl/test_08_01_etl_basics.py`.

**Check:** `pytest tests/etl/test_08_01_etl_basics.py -k test_extract`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · A strict transform

**Implement:** `transform_order(raw: dict) -> dict`

Convert a raw order row (all strings) into a typed record or raise `ValueError` with a clear message:
`order_id` (non-blank), `customer_id` (non-blank), `product_id`, `quantity` (int ≥ 1), `unit_price` (float ≥ 0), `amount` = round(q × p, 2),
`order_ts` (ISO with `Z` → aware UTC datetime), `order_date` (`YYYY-MM-DD` string), `status` (upper; must be one of PLACED/SHIPPED/DELIVERED/CANCELLED/RETURNED),
`updated_at` (datetime), `channel` (lower), `country` (upper). Lineage fields (`_source`, `_line`) are passed through if present.

Example raw row: `{"order_id": "O1", "customer_id": "C1", "product_id": "P1", "quantity": "2", "unit_price": "10.50", "order_ts": "2026-01-05T10:00:00Z", "status": "shipped", "updated_at": "2026-01-05T12:00:00Z", "channel": "WEB", "country": "IN"}`

**Input:** `raw: dict`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_transform_order_ok, test_transform_order_rejects` in `tests/etl/test_08_01_etl_basics.py`.

**Check:** `pytest tests/etl/test_08_01_etl_basics.py -k "test_transform_order_ok or test_transform_order_rejects"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Daily aggregates

**Implement:** `aggregate_daily(records: Iterable[dict]) -> list[dict]`

From typed records, compute one row per `(order_date, country)`: `orders` (distinct order_ids), `units` (Σ quantity), `revenue` (Σ amount, rounded 2), excluding `CANCELLED` orders. Sort by date then country.

**Input:** `records: Iterable[dict]`

**Expected output:** `list[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_aggregate_daily` in `tests/etl/test_08_01_etl_basics.py`.

**Check:** `pytest tests/etl/test_08_01_etl_basics.py -k test_aggregate_daily`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Wire it together with rejects and stats

**Implement:** `run_etl(input_path, output_dir) -> dict`

Use your functions: `extract` → `transform_order` (on `ValueError`, write the raw row + `_error` to `rejected.csv`) → write good rows to `clean.csv`
(datetimes as ISO strings with `Z`) → `aggregate_daily` → `daily_summary.csv`. Stream clean and rejected rows to disk as you go (don't keep raw rows in memory);
keep only what aggregation needs. Return `{"read", "loaded", "rejected", "reject_ratio" (4 decimals), "outputs": {name: path_str}}`. Run it on `datasets/csv/orders.csv`.

**Input:** `input_path, output_dir`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_run_etl, test_run_etl_real` in `tests/etl/test_08_01_etl_basics.py`.

**Check:** `pytest tests/etl/test_08_01_etl_basics.py -k "test_run_etl or test_run_etl_real"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Multi-file streaming ETL with a manifest · 🧩 Mini-project

**Implement:** `run_multi_file_etl(input_paths: list, output_dir, chunk_size: int = 500) -> dict`

Process many input files in one run. For each file: stream rows through `transform_order`, write good rows to `output_dir/clean/part-00000.csv`, `part-00001.csv`, ...
(**a new part file every `chunk_size` good rows**, across files — like Spark output parts), rejected rows to `output_dir/rejected/<source_name>` (only created if needed).
Write `output_dir/_manifest.json`: `{"files": [{"source", "read", "loaded", "rejected"}...], "parts": [{"path" (relative POSIX), "rows", "sha256"}...], "totals": {...}}` and an empty
`_SUCCESS` marker **last** (consumers wait for it). Clear any previous `clean/`/`rejected/` output first so reruns are deterministic. Return the manifest dict.

**Input:** `input_paths: list, output_dir, chunk_size: int = 500`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_run_multi_file_etl` in `tests/etl/test_08_01_etl_basics.py`.

**Check:** `pytest tests/etl/test_08_01_etl_basics.py -k test_run_multi_file_etl`

**YOUR CODE ↓**